In [ ]:
# -- Cell 1: Install dependencies ------------------------------------------
# Run once per session / Executar uma vez por sessão

# ============================================================
# CÉLULA 1 — Instalar dependências
# ============================================================

%pip install sentence-transformers transformers torch datasets -q
print("\u2713 Dependências instaladas")

In [ ]:
# -- Cell 2: Synthetic training pairs (pathway, NANDA-I diagnosis, label) ---
# Pairs generated from actual STRING enrichment pathways (Up: 9, Down: 26)
# and NANDA-I diagnoses from clinicalbert_vias_AVC.ipynb (Cell 4).
#
# Label scale:
#   >= 0.85  high relevance   (direct immune–nursing alignment)
#   0.50–0.84 moderate relevance (indirect / secondary clinical link)
#   <= 0.20  hard negative     (immune pathway x non-immune functional diagnosis)
#
# ============================================================
# CÉLULA 2 — Pares de treinamento sintéticos
# 64 pares: ~22 alta / ~22 moderada / ~20 negativos hard
# ============================================================

training_pairs = [
    ("Neutrophil degranulation",
     "Risk for infection: susceptibility to invasion and multiplication of pathogenic organisms related to immunosuppression immobility and invasive procedures (NANDA-I 00004)",
     0.92),
    ("Innate Immune System",
     "Ineffective protection: decrease in ability to guard self from internal or external threats related to altered immune response and impaired adaptive immunity (NANDA-I 00043)",
     0.91),
    ("Innate immunity",
     "Hyperthermia: core body temperature above normal diurnal range related to systemic inflammatory response and post-stroke infection (NANDA-I 00007)",
     0.88),
    ("Immune effector process",
     "Risk for infection: susceptibility to invasion and multiplication of pathogenic organisms related to immunosuppression immobility and invasive procedures (NANDA-I 00004)",
     0.87),
    ("Immune response",
     "Ineffective protection: decrease in ability to guard self from internal or external threats related to altered immune response and impaired adaptive immunity (NANDA-I 00043)",
     0.86),
    ("Secretory granule",
     "Hyperthermia: core body temperature above normal diurnal range related to systemic inflammatory response and post-stroke infection (NANDA-I 00007)",
     0.85),
    ("Neutrophil degranulation",
     "Ineffective protection: decrease in ability to guard self from internal or external threats related to altered immune response and impaired adaptive immunity (NANDA-I 00043)",
     0.9),
    ("Innate Immune System",
     "Risk for infection: susceptibility to invasion and multiplication of pathogenic organisms related to immunosuppression immobility and invasive procedures (NANDA-I 00004)",
     0.89),
    ("Immune effector process",
     "Hyperthermia: core body temperature above normal diurnal range related to systemic inflammatory response and post-stroke infection (NANDA-I 00007)",
     0.86),
    ("Innate immunity",
     "Risk for infection: susceptibility to invasion and multiplication of pathogenic organisms related to immunosuppression immobility and invasive procedures (NANDA-I 00004)",
     0.9),
    ("Lymphocyte activation",
     "Ineffective protection: decrease in ability to guard self from internal or external threats related to altered immune response and impaired adaptive immunity (NANDA-I 00043)",
     0.88),
    ("Adaptive immune response",
     "Risk for infection: susceptibility to invasion and multiplication of pathogenic organisms related to immunosuppression immobility and invasive procedures (NANDA-I 00004)",
     0.9),
    ("Unusual infection",
     "Risk for infection: susceptibility to invasion and multiplication of pathogenic organisms related to immunosuppression immobility and invasive procedures (NANDA-I 00004)",
     0.93),
    ("B cell activation",
     "Ineffective protection: decrease in ability to guard self from internal or external threats related to altered immune response and impaired adaptive immunity (NANDA-I 00043)",
     0.86),
    ("T cell differentiation",
     "Ineffective protection: decrease in ability to guard self from internal or external threats related to altered immune response and impaired adaptive immunity (NANDA-I 00043)",
     0.87),
    ("Immunoglobulin production",
     "Risk for infection: susceptibility to invasion and multiplication of pathogenic organisms related to immunosuppression immobility and invasive procedures (NANDA-I 00004)",
     0.88),
    ("Lymphocyte differentiation",
     "Ineffective protection: decrease in ability to guard self from internal or external threats related to altered immune response and impaired adaptive immunity (NANDA-I 00043)",
     0.87),
    ("Adaptive immunity",
     "Risk for infection: susceptibility to invasion and multiplication of pathogenic organisms related to immunosuppression immobility and invasive procedures (NANDA-I 00004)",
     0.91),
    ("Positive regulation of immune system process",
     "Ineffective protection: decrease in ability to guard self from internal or external threats related to altered immune response and impaired adaptive immunity (NANDA-I 00043)",
     0.87),
    ("Antigen receptor-mediated signaling pathway",
     "Risk for infection: susceptibility to invasion and multiplication of pathogenic organisms related to immunosuppression immobility and invasive procedures (NANDA-I 00004)",
     0.86),
    ("Regulation of immune response",
     "Ineffective protection: decrease in ability to guard self from internal or external threats related to altered immune response and impaired adaptive immunity (NANDA-I 00043)",
     0.86),
    ("Immunoglobulin production",
     "Ineffective protection: decrease in ability to guard self from internal or external threats related to altered immune response and impaired adaptive immunity (NANDA-I 00043)",
     0.85),
    ("Neutrophil degranulation",
     "Acute confusion: abrupt onset of reversible disturbances of consciousness attention cognition and perception related to neuroinflammation and cerebral ischemia (NANDA-I 00128)",
     0.72),
    ("Innate immunity",
     "Risk for ineffective cerebral tissue perfusion: susceptibility to decrease in cerebral tissue circulation related to cerebral ischemia edema and blood-brain barrier disruption (NANDA-I 00201)",
     0.75),
    ("Immune response",
     "Impaired tissue integrity: damage to mucous membrane corneal integumentary or subcutaneous tissues related to ischemic injury and impaired circulation (NANDA-I 00044)",
     0.7),
    ("Secretory granule",
     "Acute pain: unpleasant sensory and emotional experience associated with actual or potential tissue damage related to neurological injury and post-stroke pain (NANDA-I 00132)",
     0.62),
    ("Innate Immune System",
     "Risk for injury: susceptibility to physical damage due to sensory and motor deficit impaired balance and altered consciousness after stroke (NANDA-I 00035)",
     0.58),
    ("Extracellular region",
     "Impaired tissue integrity: damage to mucous membrane corneal integumentary or subcutaneous tissues related to ischemic injury and impaired circulation (NANDA-I 00044)",
     0.67),
    ("Cytoplasmic vesicle",
     "Risk for pressure injury: susceptibility to localized damage to skin and underlying tissue usually over bony prominence related to immobility and sensory deficit (NANDA-I 00249)",
     0.56),
    ("Lymphocyte activation",
     "Anxiety: vague uneasy feeling of discomfort or dread accompanied by autonomic response related to neurological deficit uncertainty and loss of independence (NANDA-I 00146)",
     0.55),
    ("Adaptive immune response",
     "Impaired tissue integrity: damage to mucous membrane corneal integumentary or subcutaneous tissues related to ischemic injury and impaired circulation (NANDA-I 00044)",
     0.65),
    ("Regulation of immune response",
     "Decreased intracranial adaptive capacity: intracranial fluid dynamic mechanisms compromised resulting in repeated disproportionate increases in intracranial pressure (NANDA-I 00049)",
     0.64),
    ("Positive regulation of immune system process",
     "Hyperthermia: core body temperature above normal diurnal range related to systemic inflammatory response and post-stroke infection (NANDA-I 00007)",
     0.78),
    ("Unusual infection",
     "Impaired tissue integrity: damage to mucous membrane corneal integumentary or subcutaneous tissues related to ischemic injury and impaired circulation (NANDA-I 00044)",
     0.7),
    ("CD22 mediated BCR regulation",
     "Ineffective protection: decrease in ability to guard self from internal or external threats related to altered immune response and impaired adaptive immunity (NANDA-I 00043)",
     0.72),
    ("Immunological synapse formation",
     "Ineffective protection: decrease in ability to guard self from internal or external threats related to altered immune response and impaired adaptive immunity (NANDA-I 00043)",
     0.74),
    ("Cell surface receptor signaling pathway",
     "Decreased intracranial adaptive capacity: intracranial fluid dynamic mechanisms compromised resulting in repeated disproportionate increases in intracranial pressure (NANDA-I 00049)",
     0.6),
    ("Positive regulation of T cell activation",
     "Ineffective protection: decrease in ability to guard self from internal or external threats related to altered immune response and impaired adaptive immunity (NANDA-I 00043)",
     0.79),
    ("Immune response",
     "Risk for ineffective cerebral tissue perfusion: susceptibility to decrease in cerebral tissue circulation related to cerebral ischemia edema and blood-brain barrier disruption (NANDA-I 00201)",
     0.68),
    ("Innate immunity",
     "Acute confusion: abrupt onset of reversible disturbances of consciousness attention cognition and perception related to neuroinflammation and cerebral ischemia (NANDA-I 00128)",
     0.72),
    ("Secretory granule lumen",
     "Risk for infection: susceptibility to invasion and multiplication of pathogenic organisms related to immunosuppression immobility and invasive procedures (NANDA-I 00004)",
     0.8),
    ("Neutrophil degranulation",
     "Risk for ineffective cerebral tissue perfusion: susceptibility to decrease in cerebral tissue circulation related to cerebral ischemia edema and blood-brain barrier disruption (NANDA-I 00201)",
     0.75),
    ("B cell proliferation",
     "Fatigue: overwhelming sustained sense of exhaustion and decreased capacity for physical and mental work related to neurological disease (NANDA-I 00093)",
     0.52),
    ("Immunoglobulin production",
     "Ineffective protection: decrease in ability to guard self from internal or external threats related to altered immune response and impaired adaptive immunity (NANDA-I 00043)",
     0.82),
    ("Neutrophil degranulation",
     "Disturbed sleep pattern: time-limited disruption of sleep amount and quality related to hospitalization neurological symptoms and pain (NANDA-I 00198)",
     0.05),
    ("Innate immunity",
     "Impaired physical mobility: limitation in independent purposeful physical movement related to neurological impairment and hemiplegia (NANDA-I 00085)",
     0.08),
    ("B cell receptor complex",
     "Bathing self-care deficit: impaired ability to independently perform bathing and hygiene activities related to hemiplegia and motor deficit (NANDA-I 00108)",
     0.05),
    ("Lymphocyte differentiation",
     "Constipation: decrease in normal bowel movement frequency and consistency related to immobility and neurological dysfunction (NANDA-I 00011)",
     0.06),
    ("Adaptive immune response",
     "Impaired verbal communication: decreased ability to receive process transmit and use a system of symbols related to aphasia after stroke (NANDA-I 00051)",
     0.07),
    ("T cell differentiation",
     "Dressing self-care deficit: impaired ability to dress and undress independently related to motor and sensory deficit after stroke (NANDA-I 00109)",
     0.05),
    ("Immunoglobulin production",
     "Interrupted family processes: change in family relationships and functioning related to acute illness and hospitalization of family member (NANDA-I 00060)",
     0.06),
    ("B cell activation",
     "Impaired physical mobility: limitation in independent purposeful physical movement related to neurological impairment and hemiplegia (NANDA-I 00085)",
     0.07),
    ("CD22 mediated BCR regulation",
     "Fatigue: overwhelming sustained sense of exhaustion and decreased capacity for physical and mental work related to neurological disease (NANDA-I 00093)",
     0.07),
    ("Immunological synapse formation",
     "Hopelessness: subjective state in which individual sees limited or no alternatives or personal choices and is unable to mobilize energy related to neurological disability (NANDA-I 00124)",
     0.05),
    ("B-cell antigen receptor complex-associated protein alpha/beta chain",
     "Constipation: decrease in normal bowel movement frequency and consistency related to immobility and neurological dysfunction (NANDA-I 00011)",
     0.04),
    ("Antigen receptor-mediated signaling pathway",
     "Impaired memory: inability to recall or retrieve bits of information or behavioral skills related to neurological damage from cerebral ischemia (NANDA-I 00131)",
     0.06),
    ("pre-B cell receptor complex",
     "Disturbed sleep pattern: time-limited disruption of sleep amount and quality related to hospitalization neurological symptoms and pain (NANDA-I 00198)",
     0.05),
    ("Secretory granule",
     "Caregiver role strain: difficulty in performing the family caregiver role related to complexity duration and demands of post-stroke care (NANDA-I 00061)",
     0.06),
    ("Extracellular region",
     "Impaired verbal communication: decreased ability to receive process transmit and use a system of symbols related to aphasia after stroke (NANDA-I 00051)",
     0.07),
    ("Cytoplasmic vesicle",
     "Impaired swallowing: abnormal functioning of swallowing mechanism compromising nutrition and airway protection due to neurological deficit (NANDA-I 00103)",
     0.08),
    ("Disulfide bond",
     "Impaired physical mobility: limitation in independent purposeful physical movement related to neurological impairment and hemiplegia (NANDA-I 00085)",
     0.03),
    ("Signal",
     "Powerlessness: lived experience of lack of control over a situation including a perception that one's actions do not significantly affect outcome related to stroke-induced dependency (NANDA-I 00125)",
     0.06),
    ("B cell receptor signaling pathway",
     "Fatigue: overwhelming sustained sense of exhaustion and decreased capacity for physical and mental work related to neurological disease (NANDA-I 00093)",
     0.08),
    ("Immunoreceptor tyrosine-based activation motif",
     "Constipation: decrease in normal bowel movement frequency and consistency related to immobility and neurological dysfunction (NANDA-I 00011)",
     0.04),
    ("Positive regulation of T cell activation",
     "Disturbed sleep pattern: time-limited disruption of sleep amount and quality related to hospitalization neurological symptoms and pain (NANDA-I 00198)",
     0.05),
]

print(f"\u2713 {len(training_pairs)} pares de treinamento definidos")
print(f"  Alta relevância (>= 0.85) : {sum(1 for _,_,l in training_pairs if l >= 0.85)}")
print(f"  Moderada (0.50–0.84)      : {sum(1 for _,_,l in training_pairs if 0.20 < l < 0.85)}")
print(f"  Negativos hard (<= 0.20)  : {sum(1 for _,_,l in training_pairs if l <= 0.20)}")

In [ ]:
# -- Cell 3: Prepare training dataset -------------------------------------
# InputExample wraps each (pathway, diagnosis, label) triple into the
# format expected by sentence-transformers training utilities.

# ============================================================
# CÉLULA 3 — Preparar dataset de treinamento
# ============================================================

from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

train_examples = [
    InputExample(texts=[p, d], label=float(l))
    for p, d, l in training_pairs
]

# batch_size=16 adequado para GPU com 6GB+ VRAM ou CPU (mais lento)
# batch_size=16 suitable for GPU with 6GB+ VRAM or CPU (slower)
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)
print(f"\u2713 {len(train_examples)} exemplos de treinamento carregados")

In [ ]:
# -- Cell 4: Load base model and define loss ------------------------------
# CosineSimilarityLoss trains the model to match predicted cosine similarity
# to the float labels in each InputExample. It directly optimizes the
# embedding space for the clinical-stroke domain.

# ============================================================
# CÉLULA 4 — Carregar modelo base e definir função de loss
# CosineSimilarityLoss: otimiza diretamente a similaridade coseno
# entre pares, alinhando o espaço latente ao domínio AVC × NANDA-I
# ============================================================

model = SentenceTransformer('pritamdeka/S-PubMedBert-MS-MARCO-SCIFACT')

# CosineSimilarityLoss: otimiza diretamente a similaridade coseno
# entre pares, alinhando o espaço latente aos labels do domínio
train_loss = losses.CosineSimilarityLoss(model)
print("\u2713 Modelo base e loss carregados")

In [ ]:
# -- Cell 5: Fine-tuning --------------------------------------------------
# Runs supervised fine-tuning for 5 epochs with linear warmup.
# The fine-tuned model is saved to outputs/sbert_nanda_finetuned/
# and can be loaded directly via SentenceTransformer(path).

# ============================================================
# CÉLULA 5 — Fine-tuning supervisionado
# epochs=5, warmup_steps=10, output salvo em outputs/sbert_nanda_finetuned/
# ============================================================

%pip install -q -U "accelerate>=1.1.0" "transformers[torch]"

import os
import accelerate
import transformers

print(f"✓ accelerate: {accelerate.__version__}")
print(f"✓ transformers: {transformers.__version__}")

os.makedirs('outputs/sbert_nanda_finetuned', exist_ok=True)

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=5,
    warmup_steps=10,
    output_path='outputs/sbert_nanda_finetuned',
    show_progress_bar=True
)
print("\u2713 Fine-tuning concluído")
print("   Modelo salvo em: outputs/sbert_nanda_finetuned/")

In [ ]:
# -- Cell 6: Quick validation --------------------------------------------
# 5 held-out pairs (not seen during training) are used to verify that
# the fine-tuned model correctly separates relevant from irrelevant pairs.
# Expected: positive pairs (expected=1) score >= 0.5;
#           negative pairs (expected=0) score <  0.5.

# ============================================================
# CÉLULA 6 — Validação rápida com pares não vistos no treino
# ============================================================

from sentence_transformers import util
import numpy as np

model_ft = SentenceTransformer('outputs/sbert_nanda_finetuned')

# 5 pares de validação não vistos no treino / 5 held-out validation pairs
val_pairs = [
    ("Innate Immune System",
     "Ineffective protection: decrease in ability to guard self from internal or external threats related to altered immune response and impaired adaptive immunity (NANDA-I 00043)",
     1),
    ("Adaptive immunity",
     "Risk for infection: susceptibility to invasion and multiplication of pathogenic organisms related to immunosuppression immobility and invasive procedures (NANDA-I 00004)",
     1),
    ("Secretory granule",
     "Impaired physical mobility: limitation in independent purposeful physical movement related to neurological impairment and hemiplegia (NANDA-I 00085)",
     0),
    ("B cell proliferation",
     "Disturbed sleep pattern: time-limited disruption of sleep amount and quality related to hospitalization neurological symptoms and pain (NANDA-I 00198)",
     0),
    ("Neutrophil degranulation",
     "Hyperthermia: core body temperature above normal diurnal range related to systemic inflammatory response and post-stroke infection (NANDA-I 00007)",
     1),
]

print("Validação (modelo fine-tuned):")
for p, d, expected in val_pairs:
    ep    = model_ft.encode(p, convert_to_tensor=True)
    ed    = model_ft.encode(d, convert_to_tensor=True)
    score = float(util.cos_sim(ep, ed))
    status = "\u2713" if (score >= 0.5) == bool(expected) else "\u2717"
    print(f"  {status} {score:.4f}  [{p[:35]:35s}] \u00d7 [{d[:55]}...]")

## Integração com o notebook principal / Integration with main notebook

Após o fine-tuning concluir, o modelo salvo em `outputs/sbert_nanda_finetuned/` pode substituir o modelo base no pipeline principal.

After fine-tuning completes, the saved model in `outputs/sbert_nanda_finetuned/` can replace the base model in the main pipeline.

---

**Em `clinicalbert_vias_AVC.ipynb`, Célula 2**, substitua:

```python
model_sbert = SentenceTransformer('pritamdeka/S-PubMedBert-MS-MARCO-SCIFACT')
```

por:

```python
model_sbert = SentenceTransformer('outputs/sbert_nanda_finetuned')
```

Execute o notebook principal novamente **a partir da Célula 5** (geração de embeddings) para propagar o novo modelo pelo pipeline de similaridade, heatmaps e exportação.

Re-run the main notebook **starting from Cell 5** (embedding generation) to propagate the new model through the similarity, heatmap, and export pipeline.